# Joint multi-sample deconvolution with named effects

Fit sections on a common feature panel with structured profile deviations. One named effect
represents flat groups; a parent and child represent two nested levels. The model centers
parent deviations across parents and child deviations within each parent. Effective profiles
remain nonnegative and normalized. Graphs remain within physical sections.

Supply `samples.h5ad` with counts, unique spot IDs, coordinates in `obsm["spatial"]`, and
`section_id` plus your design columns in `obs`. Each physical section belongs to one profile
group; several sections may share that group. Sample, donor and visit are example effect names.

## Data links

- CRC liver-section measurements: [GEO GSE294385](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE294385).
- Spot-level annotations: [CRC_micromets_ST](https://github.com/yliuup/CRC_micromets_ST), `Meta_data` directory.

## Example figure

Archived CRC example: component maps across 12 liver sections using patient and
within-patient condition effects. The cells below show how to configure and fit your own design.

![Jointly fitted CRC component maps across twelve sections](../docs/source/_static/figures/crc_480_sections.png)


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR`, the input filenames and `K` below.
Supply your own `.h5ad` files with unique spot and feature IDs. RNA fitting uses raw counts;
set `COUNTS_LAYER="counts"` when they are stored in a layer. Protein inputs should contain
your prepared nonnegative assay measurements. Each fitted spot needs positive total signal.

Running the fitting cells trains a new model. `DEVICE` accepts `"auto"`, `"cpu"` or `"cuda"`.
The study's reference-free and joint analyses used GPUs. Inspect the returned diagnostics
to determine whether your fit converged. Large inputs require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
DEVICE = "auto"
VERBOSE = True
MAX_ITER = 20000
set_seed(1, deterministic=True)

K = 10  # Choose the component number for your data.
effects = {"sample": {"column": "sample_id", "penalty": 10.}}


## 2. Read sections and design metadata


In [ ]:
adata = ad.read_h5ad(DATA_DIR / "samples.h5ad")
y = adata.to_df(layer=COUNTS_LAYER)
metadata = adata.obs.copy()
xy = np.asarray(adata.obsm["spatial"])
design_columns = list(dict.fromkeys(["section_id"] + [spec["column"] for spec in effects.values()]))
display(metadata[design_columns].drop_duplicates())


## 3. Choose a design and fit

The setup uses a flat `sample_id` effect. For visits nested within donors, replace it there with:

```python
effects = {
    "donor": {"column": "donor_id", "penalty": 10.},
    "visit": {"column": "timepoint", "parent": "donor", "penalty": 20.},
}
```

`column` names a metadata column; `parent` names another effect key. The same visit name in two
donors has separate within-donor deviations. A parent with one observed child has zero child
deviation. Choose penalties for your analysis. The interface supports one flat effect or two
nested effects. The fit initializes from your input counts; no saved starting factors are needed.


In [ ]:
from spDecon.multisample import fit_multisample
result = fit_multisample(y, metadata, K, effects=effects, coordinates=xy,
    spatial_weight=.05, random_state=1, device=DEVICE, max_iter=MAX_ITER, verbose=VERBOSE)
p = result.proportions
shared = result.extra["shared_profiles"]
display(result.diagnostics)
for name in result.extra["effect_names"]:
    print(name, result.extra[f"effect_delta_{name}"].shape)
    display(pd.DataFrame(result.extra[f"effect_labels_{name}"],
        columns=result.extra[f"effect_columns_{name}"]))


## 4. Display every section


In [ ]:
sections = pd.unique(metadata.section_id)
fig, axes = plt.subplots(int(np.ceil(len(sections)/4)), 4,
    figsize=(12, 3*int(np.ceil(len(sections)/4))), squeeze=False)
for ax, section in zip(axes.ravel(), sections):
    keep = metadata.section_id.eq(section).to_numpy()
    spatial(ax, xy[keep], p[keep].argmax(axis=1), str(section), categorical=True, size=10)
for ax in axes.ravel()[len(sections):]:
    ax.axis("off")
legend(fig, [f"C{k+1}" for k in range(K)], columns=min(K, 6), y=.01)
fig.subplots_adjust(bottom=.14)
save_figure(fig, "multisample_sections")


## 5. Inspect shared profiles and export results

The ranking below divides each shared-profile value by its mean across components. It helps
interpret fitted profiles; it is not a differential-expression test. Effect arrays and their
metadata labels are exported together so that deviation rows remain identifiable.


In [ ]:
enrichment = shared / np.maximum(shared.mean(axis=0), 1e-12)
rows = []
for k in range(K):
    genes = adata.var_names[np.argsort(enrichment[k])[-5:][::-1]]
    rows.append({"component": f"C{k+1}", "profile_markers": ", ".join(genes), "mean_weight": p[:,k].mean()})
display(save_table(pd.DataFrame(rows), "multisample_component_summary"))
out = ROOT / "outputs/multisample"
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame(p, index=adata.obs_names, columns=[f"C{k+1}" for k in range(K)]).to_csv(out / "weights.csv")
arrays = {key: value for key, value in result.extra.items() if key.startswith("effect_")}
np.savez_compressed(out / "effects.npz", shared_profiles=shared, **arrays)
